In [2]:
!python -m pip install numpy icecream

In [10]:
import numpy as np
import random
import heapq
import re
from pprint import pprint
from math import ceil
from functools import total_ordering
from collections import defaultdict
from icecream import ic

In [2]:
TRACE_FILE="./skew_uniform_3.0_300.csv"
# TRACE_FILE="./skew_uniform_16.0_180.csv"
servers = [0,1,2,3]
backend="system"
step = 30

In [3]:
@total_ordering
class Request:
    def __init__(self, req_id, model_dir, adapter_dir, prompt, prompt_len, output_len, req_time):
        self.req_id = req_id
        self.model_dir = model_dir 
        self.adapter_dir = adapter_dir
        self.prompt = prompt
        self.prompt_len = prompt_len
        self.output_len = output_len
        self.req_time = req_time

    def __repr__(self):
        return f"req_id={self.req_id}, " \
               f"model_dir={self.model_dir}, adapter_dir={self.adapter_dir}, " \
               f"prompt_len={self.prompt_len}, output_len={self.output_len}, " \
               f"req_time={self.req_time}"

    def __eq__(self, other):
        return self.req_id == other.req_id

    def __lt__(self, other):
        return self.req_time < other.req_time

def dummy_prompt(prompt_len):
    return "Hello " * prompt_len


In [4]:
def read_requests(trace_file):
    requests = []
    adapter_dirs = set()
    with open(trace_file, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:
            elements = line.split(",")
            requests.append(
                Request(
                    req_id=int(elements[0]),
                    model_dir=elements[1],
                    adapter_dir=elements[2],
                    prompt=dummy_prompt(int(elements[3])),
                    prompt_len=int(elements[3]),
                    output_len=int(elements[4]),
                    req_time=float(elements[5]),
                )
            )
            # requests.append((int(elements[0]),elements[1],elements[2],int(elements[3]),int(elements[4]),float(elements[5])))
            adapter_dirs.add(elements[2])
    requests.sort(key=lambda r: r.req_time)
    return list(adapter_dirs), requests

In [5]:
adapter_dirs, requests = read_requests(trace_file=TRACE_FILE)
avg_prompt_len = np.mean([req.prompt_len for req in requests])
avg_output_len = np.mean([req.output_len for req in requests])
avg_len = np.mean([req.prompt_len + req.output_len for req in requests])
print(
    "num_adapters",
    len(adapter_dirs),
    "num_requests",
    len(requests),
    "avg_len:",
    avg_len,
    "avg_prompt_len:",
    avg_prompt_len,
    "avg_output_len:",
    avg_output_len,
)

num_adapters 25 num_requests 900 avg_len: 628.0 avg_prompt_len: 500.0 avg_output_len: 128.0


In [6]:
# step_idx = 0
# last_time = requests[0]
# for req in requests:
#     if req.req_time > last_time.req_time // 1 + step:
#         demand_tps = {a: 0 for a in adapter_dirs}
#         index = requests.index(last_time)
#         while requests[index].req_time < req.req_time:
#             # rank = int(re.search(r'rank-(\d+)', requests[index].adapter_dir).group(1))
#             demand_tps[requests[index].adapter_dir] = (
#                 demand_tps.get(requests[index].adapter_dir)
#                 + (
#                     requests[index].prompt_len
#                     + requests[index].output_len
#                 )
#                 / step
#             )
#             index += 1
#         adapter_demand = []
#         for adapter, tps in demand_tps.items():
#             rank = int(re.search(r"rank-(\d+)", adapter).group(1))
#             adapter_demand.append((rank, tps, adapter))  # tps here is expected tps
#         adapter_demand.sort(reverse=True)

#         # server_tps = {8: 2400, 16: 2100, 32: 1900, 64:1700, 128:1600} # operating point, fn of max rank, old NC24ads-hipri 8xA100 40GB
#         server_tps = {
#             8: 2725,
#             16: 2700,
#             32: 2675,
#             64: 2625,
#             128: 2525,
#         }  # operating point, fn of max rank, 4xA100 80GB

#         adapter_groups = [[] for _ in servers]
#         num_servers = len(servers)
#         server_occupied_tps = [0] * num_servers
#         server_max_rank = [0] * num_servers
#         adapters_placed = [False] * len(adapter_demand)

#         with open("allocation_log.txt", "a") as f:
#             f.write("\n\n************************************")
#             f.write(f"step {step_idx} @ time {last_time.req_time} to {req.req_time}:")

#         def is_compatible(
#             group_idx,
#             tuple,
#             server_max_rank=server_max_rank,
#             server_occupied_tps=server_occupied_tps,
#             scale=1,
#         ):
#             """
#             Check if the given adapter tuple can fit within the group
#             An adapter can fit if it is within the tps limit
#             The tps limit depends on the max rank of the allocated adapters to this server
#             """
#             rank, tps, _ = tuple
#             max_rank = max(rank, server_max_rank[group_idx])
#             tps = server_occupied_tps[group_idx] + tuple[1]
#             return tps <= (server_tps[max_rank] * scale)

#         # * checking compatibility
#         rank_instance_budget = [(rank, sum(tps for r, tps, _ in adapter_demand if r == rank) / rank_max_tps) for rank, rank_max_tps in server_tps.items()]
#         sorted_budgets = sorted(rank_instance_budget, key=lambda x: x[1], reverse=True)
#         assert sum(budget for _, budget in rank_instance_budget) <= num_servers, "Exceeded server budget"

#         # * rounding
#         rounded_budgets = [(rank, round(budget)) for rank, budget in sorted_budgets]
#         sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)
#         if sum_rounded_off_budgets < num_servers:
#             idx = 0
#             while sum_rounded_off_budgets < num_servers and idx < len(rounded_budgets):
#                 rounded_budgets[idx] = (rounded_budgets[idx][0], ceil(sorted_budgets[idx][1]))
#                 idx += 1
#                 sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)

#         # * balanced allocation within assigned instances
#         adapters_with_assigned_instances = [x for x in rounded_budgets if x[1] > 0]
#         current_server = 0
#         for rank, num_assigned_instances in adapters_with_assigned_instances:
#             l = current_server
#             r = current_server + num_assigned_instances
#             server_heap = [(server_occupied_tps[i], i) for i in range(l, r)]
#             heapq.heapify(server_heap)
#             for adapter_idx, adapter in enumerate(adapter_demand):
#                 if adapter[0] == rank:
#                     while server_heap:
#                         occupancy, least_occupied_server = heapq.heappop(server_heap)
#                         if is_compatible(least_occupied_server, adapter):
#                             adapter_groups[least_occupied_server].append(adapter)
#                             server_occupied_tps[least_occupied_server] += adapter[1]
#                             adapters_placed[adapter_idx] = True
#                             server_max_rank[least_occupied_server] = max(server_max_rank[least_occupied_server], rank)
#                             heapq.heappush(server_heap, (server_occupied_tps[least_occupied_server], least_occupied_server))
#                             break
                
#             current_server += num_assigned_instances

#         # * leftovers
#         for adapter_idx, adapter in enumerate(adapter_demand):
#             if not adapters_placed[adapter_idx]:
#                 least_occupied_server = min(
#                     (i for i in range(num_servers) if server_max_rank[i] >= adapter[0]),
#                     key=lambda x: server_occupied_tps[x],
#                     default=None
#                 )
#                 if least_occupied_server is not None and is_compatible(least_occupied_server, adapter):
#                     adapter_groups[least_occupied_server].append(adapter)
#                     server_occupied_tps[least_occupied_server] += adapter[1]
#                     adapters_placed[adapter_idx] = True
#                     continue

#                 # we could not find a server with rank >= this adapters rank
#                 # need to colocate with a lower rank
#                 # TODO: better logic here - search through the closest ranks first and stop if we can fit
#                 new_least_occupied_server = min(range(num_servers), key=lambda x: server_occupied_tps[x])
#                 if is_compatible(new_least_occupied_server, adapter):
#                     with open("allocation_log.txt", "a") as f:
#                         f.write(
#                             f"{last_time} Adapter {adapter[2]} with rank {adapter[0]}, tps {adapter[1]} could not be placed in a server with rank >= {adapter[0]}, placing in server {servers[new_least_occupied_server]} with max rank {server_max_rank[new_least_occupied_server]}\n"
#                         )
#                     adapter_groups[new_least_occupied_server].append(adapter)
#                     server_occupied_tps[new_least_occupied_server] += adapter[1]
#                     adapters_placed[adapter_idx] = True
#                     server_max_rank[new_least_occupied_server] = max(server_max_rank[new_least_occupied_server], adapter[0])


#         with open("allocation_log.txt", "a") as f:
#             f.write(f"\n{last_time} Adapter groups:\n")
#             for i, group in enumerate(adapter_groups):
#                 f.write(
#                     f"  Server {servers[i]}: {[(adapter, tps) for _, tps, adapter in group]}\n"
#                 )
#                 f.write(
#                     f"  Server {servers[i]} total tps: {server_occupied_tps[i]} max tps: {server_tps[server_max_rank[i]]}\n"
#                 )
#                 f.write(
#                     f"  Server {servers[i]} max tps: {server_tps[server_max_rank[i]]}\n"
#                 )
#                 f.write(
#                     f"  Server {servers[i]} util: {server_occupied_tps[i]/server_tps[server_max_rank[i]]}\n"
#                 )
#                 f.write(
#                     f"  Server {servers[i]} max rank: {server_max_rank[i]}\n"
#                 )
#             f.write("************************************\n\n")

#         server_map = {}
#         for i, server in enumerate(servers):
#             for _, _, adapter in adapter_groups[i]:
#                 server_map[adapter] = server
#         step_idx += 1
#         last_time = req

In [7]:
def compare_with_prev_alloc(
    adapter_groups,
    prev_adapter_groups,
    assigned_instances_per_rank,
    prev_assigned_instances_per_rank,
    adapter_to_tps=None,
):
    """
    Compares the current and previous allocations to identify any changes in server assignments.
    assigned_instances_per_rank: {rank: [server_indices]}
    adapter_groups: [(adapter_name, routing_probability),]
    probability_delta: float, if the routing probability changes by more than this value, we consider it a different server
    """
    server_rename_map = {}  # {old_server: new_server}
    
    for rank, prev_servers in prev_assigned_instances_per_rank.items():
        if rank in assigned_instances_per_rank:
            curr_servers = assigned_instances_per_rank[rank]
            if len(prev_servers) == len(curr_servers):
                for old_server, new_server in zip(prev_servers, curr_servers):
                    server_rename_map[old_server] = new_server
            else:
                for prev_server in prev_servers:
                    old_adapters = set(adapter for adapter, _ in prev_adapter_groups[prev_server])
                    for curr_server in curr_servers:
                        new_adapters = set(adapter for adapter, _ in adapter_groups[curr_server])
                        intersection_adapters_sum_tps = sum(adapter_to_tps.get(adapter, 0) for adapter in old_adapters & new_adapters)
                        union_adapters_sum_tps = sum(adapter_to_tps.get(adapter, 0) for adapter in old_adapters | new_adapters)
                        # if len(old_adapters & new_adapters) / len(old_adapters | new_adapters) > 0.5:
                        #     server_rename_map[curr_server] = prev_server
                        #     break
                        if union_adapters_sum_tps > 0 and intersection_adapters_sum_tps / union_adapters_sum_tps > 0.5:
                            server_rename_map[curr_server] = prev_server
                            break

    # for rank, curr_servers in assigned_instances_per_rank.items():
    #     prev_servers = prev_assigned_instances_per_rank.get(rank, [])
    #     if prev_servers:
    #         if set(curr_servers) == set(prev_servers):
    #             # same servers
    #             for curr_server in curr_servers:
    #                 server_rename_map[curr_server] = curr_server
    #         else:
    #             # renamed to different servers, find which server is which based on the adapters
    #             for curr_server in curr_servers:
    #                 change_in_tps = 0
    #                 adapters = [curr_adapter for curr_adapter, curr_prob in adapter_groups[curr_server]]
    #                 adapter_to_prob = {curr_adapter: curr_prob for curr_adapter, curr_prob in adapter_groups[curr_server]}
    #                 for prev_server in prev_servers:
    #                     # if the adapter is in both servers, add the change in probability
    #                     for prev_adapter, prev_prob in adapter_groups[prev_server]:
    #                         if prev_adapter in adapters:
    #                             change_in_tps += abs(adapter_to_prob[prev_adapter] - prev_prob) * adapter_to_tps.get(prev_adapter, 0)
    #                         else:
    #                             change_in_tps += prev_prob * adapter_to_tps.get(prev_adapter, 0)
                                
    #                 if change_in_tps < tps_delta:
    #                     server_rename_map[curr_server] = prev_server
    #                     break
        
    return server_rename_map

In [8]:
def ensure_all_placed(adapter_groups):
    """
    Ensure that the probability of adapter placements sums to 1 for each adapter
    """

    utils = {}
    for group in adapter_groups:
        for adapter, util in group:
            if adapter in utils:
                utils[adapter] += util
            else:
                utils[adapter] = util

    for adapter, util in utils.items():
        assert abs(util - 1) < 1e-3, f"Adapter {adapter} not fully placed, util={util}"

In [9]:
def reset_allocation_log():
    with open("./allocation_log.txt", "w") as f:
        f.write("")

In [12]:
step_idx = 0
last_time = requests[0]
prev_alloc = None
prev_rank_assigned_instances = None
debug = True

for req in requests:
    if req.req_time > last_time.req_time // 1 + step:
        demand_tps = {a: 0 for a in adapter_dirs}
        index = requests.index(last_time)
        while requests[index].req_time < req.req_time:
            # rank = int(re.search(r'rank-(\d+)', requests[index].adapter_dir).group(1))
            demand_tps[requests[index].adapter_dir] = (
                demand_tps.get(requests[index].adapter_dir)
                + (
                    requests[index].prompt_len
                    + requests[index].output_len
                )
                / step
            )
            index += 1
        adapter_demand = []
        for adapter, tps in demand_tps.items():
            rank = int(re.search(r"rank-(\d+)", adapter).group(1))
            adapter_demand.append((rank, tps, adapter))  # tps here is expected tps
        adapter_demand.sort(reverse=True)

        rank_wise_demand = {}
        for rank, tps, adapter in adapter_demand:
            if rank not in rank_wise_demand:
                rank_wise_demand[rank] = 0
            rank_wise_demand[rank] += tps

        # server_tps = {8: 2400, 16: 2100, 32: 1900, 64:1700, 128:1600} # operating point, fn of max rank, old NC24ads-hipri 8xA100 40GB
        server_tps = {
            8: 2725,
            16: 2700,
            32: 2675,
            64: 2625,
            128: 2525,
        }  # operating point, fn of max rank, 4xA100 80GB

        rank_instance_demand = {}
        for rank, tps in rank_wise_demand.items():
            rank_instance_demand[rank] = tps / server_tps[rank]
        
        total_instance_demand = sum(rank_instance_demand.values())

        target_util = total_instance_demand / len(servers)
        assert target_util <= 1, f"Target utilization exceeds 1, need more servers: {target_util}"

        rank_instance_demand_tuples = [(demand, rank) for rank, demand in rank_instance_demand.items()]

        adapter_groups = [[] for _ in servers]
        num_servers = len(servers)
        server_occupied_tps = [0] * num_servers
        server_max_rank = [0] * num_servers
        adapters_placed = [False] * len(adapter_demand)

        with open("allocation_log.txt", "a") as f:
            f.write("\n\n************************************")
            f.write(f"step {step_idx} @ time {last_time.req_time} to {req.req_time}:")

        # * checking compatibility
        rank_instance_budget = [(rank, sum(tps for r, tps, _ in adapter_demand if r == rank) / rank_max_tps) for rank, rank_max_tps in server_tps.items()]
        sorted_budgets = sorted(rank_instance_budget, key=lambda x: x[1], reverse=True)
        assert sum(budget for _, budget in rank_instance_budget) <= num_servers, "Exceeded server budget"
        if debug:
            ic(sorted_budgets, target_util)
        # * rounding
        rounded_budgets = [(budget, rank, round(budget/target_util))
                           for rank, budget in sorted_budgets if round(budget/target_util) > 0]
        rounded_budgets.sort(reverse=True, key=lambda x: (x[0]/x[2], x[1]))
        if debug:
            ic(rounded_budgets)
        zero_budgets = [(budget, rank, round(budget/target_util))
                           for rank, budget in sorted_budgets if round(budget/target_util) == 0]
        
        sum_rounded_off_budgets = sum(budget for _, _, budget in rounded_budgets)
        while sum_rounded_off_budgets < num_servers:
            print("Increasing instances")
            first = rounded_budgets[0]
            rounded_budgets = [(first[0], first[1], first[2]+1)] + rounded_budgets[1:].copy()
            rounded_budgets.sort(reverse=True, key=lambda x: (x[0]/x[2], x[1]))
            sum_rounded_off_budgets = sum(budget for _, _, budget in rounded_budgets)
        rounded_budgets.sort(key=lambda x: x[1])
        sum_rounded_off_budgets = sum(budget for _, _, budget in rounded_budgets)
        while sum_rounded_off_budgets > num_servers:
            print("Decreasing instances")
            first = rounded_budgets[0]
            assert first[2] > 0, "Cannot reduce instances further"
            rounded_budgets = [(first[0], first[1], first[2] - 1)] + rounded_budgets[1:].copy()
            rounded_budgets.sort(key=lambda x: x[1])
            sum_rounded_off_budgets = sum(budget for _, _, budget in rounded_budgets)
        ic(rounded_budgets)
        # if sum_rounded_off_budgets < num_servers:
        #     idx = 0
        #     while sum_rounded_off_budgets < num_servers and idx < len(rounded_budgets):
        #         rounded_budgets[idx] = (rounded_budgets[idx][0], ceil(sorted_budgets[idx][1]))
        #         idx += 1
        #         sum_rounded_off_budgets = sum(budget for _, budget in rounded_budgets)

        # * balanced allocation within assigned instances
        ranks_with_assigned_instances = [(x[1], x[2]) for x in rounded_budgets]
        rank_assigned_instances = {}
        ranks_with_zero_instances = [x[1] for x in zero_budgets]
        ic("assigned", ranks_with_assigned_instances)
        ic("zero", ranks_with_zero_instances)
        last_used_server = 0
        server_util = [0] * num_servers
        leftovers = []
        for rank in ranks_with_zero_instances:
            leftovers.extend([(idx, adapter, 1.0) for idx, adapter in enumerate(adapter_demand) if adapter[0] == rank])
        for rank, budget in ranks_with_assigned_instances:
            # assign adapters of rank to num_instances greedily
            # maybe sort in descending tps order and assign fractionally from the left

            # get all adapters of this rank
            adapters_of_rank = [(idx, adapter) for idx, adapter in enumerate(adapter_demand) if adapter[0] == rank]
            adapters_of_rank.sort(reverse=True, key=lambda x: x[1][1])  # sort by tps descending
            ic(adapters_of_rank)
            servers_used = 0
            rank_assigned_instances[rank] = list(range(last_used_server, last_used_server + budget))
            for adapter_idx, adapter in adapters_of_rank:
                tps, adapter_name = adapter[1], adapter[2]
                expected_util = tps / server_tps[rank]
                _expected_util = expected_util
                while expected_util > 1e-3 and servers_used < budget:
                    #assign as much as possible to this server
                    if servers_used >= budget:
                        ic(servers_used, budget, adapter_idx, adapter)
                        raise Exception("Ran out of servers")
                    server_idx = last_used_server + servers_used
                    # assign to this server
                    max_addable_util = min(target_util - server_util[server_idx], expected_util)
                    adapter_groups[server_idx].append([adapter_name, max_addable_util/_expected_util])
                    server_occupied_tps[server_idx] += max_addable_util * server_tps[rank]
                    
                    # adapters_placed[adapter_idx] = True
                    server_max_rank[server_idx] = max(server_max_rank[server_idx], rank)
                    server_util[server_idx] += max_addable_util
                    if server_util[server_idx] >= target_util:
                        servers_used += 1
                    expected_util -= max_addable_util
                if expected_util > 1e-3:
                    leftovers.append((adapter_idx, adapter, expected_util/_expected_util))
            last_used_server += budget
        
        if debug:
            for adapter_group in adapter_groups:
                ic(adapter_group)
            ic(leftovers)

        #* leftovers
        space_left = (target_util * num_servers) - sum(server_util)
        demand_left = 0
        for _, adapter_tuple, util_fraction in leftovers:
            demand_left += (adapter_tuple[1] * util_fraction)/server_tps[adapter_tuple[0]]
        assert demand_left - space_left < 1e-3, "Leftover demand does not fit in remaining space"

        if debug:
            ic("before leftovers")
            for server_idx in range(num_servers):
                ic(server_idx, server_util[server_idx], target_util)
            ic("space left", space_left)
            ic("demand left", demand_left)

        leftovers.sort(reverse=True, key=lambda x: (x[1][1]))  # sort by tps descending
        for _, adapter_tuple, util_fraction in leftovers:
            adapter_rank, adapter_demand_tps, adapter_name = adapter_tuple
            _expected_util = adapter_demand_tps / server_tps[adapter_rank]
            adapter_demand_tps *= util_fraction
            expected_util = adapter_demand_tps / server_tps[adapter_rank]
            # go through the servers with higher max rank and fit the fractional demand until target utilization
            server_idx = 0
            allocated_adapter = False
            while expected_util > 1e-3 and server_idx < num_servers:
                if server_max_rank[server_idx] >= adapter_rank and server_util[server_idx] < target_util:
                    max_addable_util = min(target_util - server_util[server_idx], expected_util)
                    adapter_groups[server_idx].append([adapter_name, max_addable_util/_expected_util])
                    server_occupied_tps[server_idx] += max_addable_util * server_tps[adapter_rank]
                    server_max_rank[server_idx] = max(server_max_rank[server_idx], adapter_rank)
                    server_util[server_idx] += max_addable_util
                    expected_util -= max_addable_util
                server_idx += 1

            if expected_util == 0:
                allocated_adapter = True

            if not allocated_adapter:
                server_idx = 0
                # we could not find a server with rank >= this adapters rank
                # need to colocate with a lower rank
                while expected_util > 1e-3 and server_idx < num_servers:
                    ic(server_idx, server_util[server_idx], target_util, expected_util)
                    if server_util[server_idx] < target_util:
                        max_addable_util = min(target_util - server_util[server_idx], expected_util)
                        adapter_groups[server_idx].append([adapter_name, max_addable_util/_expected_util])
                        server_occupied_tps[server_idx] += max_addable_util * server_tps[adapter_rank]
                        server_max_rank[server_idx] = max(server_max_rank[server_idx], adapter_rank)
                        server_util[server_idx] += max_addable_util
                        expected_util -= max_addable_util
                    server_idx += 1

            if expected_util > 1e-3:
                ic(total_instance_demand, num_servers, target_util)
                raise Exception(f"Could not allocate adapter {adapter_name} with rank {adapter_rank} and tps {adapter_demand_tps}, leftover util {expected_util}")


        with open("allocation_log.txt", "a") as f:
            f.write(f"\n{last_time} Adapter groups:\n")
            for i, group in enumerate(adapter_groups):
                f.write(
                    f"  Server {servers[i]}: {[(adapter, used_util) for adapter, used_util in group]}\n"
                )
                # f.write(
                #     f"  Server {servers[i]} total tps: {server_occupied_tps[i]} max tps: {server_tps[server_max_rank[i]]}\n"
                # )
                f.write(
                    f"  Server {servers[i]} max tps: {server_tps.get(server_max_rank[i], 0)}\n"
                )
                f.write(
                    f"  Server {servers[i]} util: {server_util[i]}\n"
                )
                f.write(
                    f"  Server {servers[i]} max rank: {server_max_rank[i]}\n"
                )
            f.write("************************************\n\n")

        ensure_all_placed(adapter_groups)
        print(f"All adapters placed successfully for step {step_idx} from time {last_time.req_time} to {req.req_time}")

        #* compare with last iteration
        server_rename_map = None
        if prev_alloc is not None and prev_rank_assigned_instances is not None:
            adapter_to_tps = {adapter: tps for _, tps, adapter in adapter_demand}
            server_rename_map = compare_with_prev_alloc(
                adapter_groups=adapter_groups,
                prev_adapter_groups=prev_alloc,
                assigned_instances_per_rank=rank_assigned_instances,
                prev_assigned_instances_per_rank=prev_rank_assigned_instances,
                adapter_to_tps=adapter_to_tps,
            )
            if debug:
                with open("allocation_log.txt", "a") as f:
                    f.write(f"Prev rank assigned instances (step {step_idx - 1}): {prev_rank_assigned_instances}\n")
                    f.write(f"Curr rank assigned instances (step {step_idx}): {rank_assigned_instances}\n")
            if server_rename_map:
                print("Server renames detected:", server_rename_map)
                with open("allocation_log.txt", "a") as f:
                    f.write(f"Server renames detected: {server_rename_map}\n")

                if debug:
                    print(f"Prev rank assigned instances (step {step_idx - 1}):", prev_rank_assigned_instances)
                    print(f"Curr rank assigned instances (step {step_idx}):", rank_assigned_instances)
                    
        server_map = defaultdict(list) # adapter -> [server1, server2, ...]
        probability_sum = defaultdict(list) # adapter -> [prob of server 1, prob of server 1 + prob of server 2, ...]
        for i, server in enumerate(servers):
            for adapter, util in adapter_groups[i]:
                if not server_rename_map or server not in server_rename_map.keys():
                    server_map[adapter].append(server)
                    probability_sum[adapter].append(probability_sum[adapter][-1] + util if probability_sum[adapter] else util)
                else:
                    server_map[adapter].append(server_rename_map[server])
        print(server_map)
        print(probability_sum)

        prev_alloc = adapter_groups.copy()
        prev_rank_assigned_instances = rank_assigned_instances.copy()
        step_idx += 1
        last_time = req

ic| sorted_budgets: [(128, 0.4891353135313531),
                     (16, 0.14730864197530866),
                     (32, 0.10173208722741432),
                     (8, 0.061455657492354744),
                     (64, 0.03987301587301587)]
    target_util: 0.20987617902486166
ic| rounded_budgets: [(0.4891353135313531, 128, 2), (0.14730864197530866, 16, 1)]
ic| rounded_budgets: [(0.14730864197530866, 16, 1), (0.4891353135313531, 128, 3)]


ic| 'assigned', ranks_with_assigned_instances: [(16, 1), (128, 3)]
ic| 'zero', ranks_with_zero_instances: [32, 8, 64]


Increasing instances


ic| adapters_of_rank: [(15, (16, 104.66666666666667, 'dummy-lora-7b-rank-16-3')),
                       (16, (16, 104.66666666666667, 'dummy-lora-7b-rank-16-2')),
                       (17, (16, 83.73333333333333, 'dummy-lora-7b-rank-16-1')),
                       (18, (16, 83.73333333333333, 'dummy-lora-7b-rank-16-0')),
                       (19, (16, 20.933333333333334, 'dummy-lora-7b-rank-16-4'))]
ic| adapters_of_rank: [(0, (128, 355.8666666666667, 'dummy-lora-7b-rank-128-1')),
                       (1, (128, 293.06666666666666, 'dummy-lora-7b-rank-128-2')),
                       (2, (128, 272.1333333333333, 'dummy-lora-7b-rank-128-3')),
                       (3, (128, 188.4, 'dummy-lora-7b-rank-128-0')),
                       (4, (128, 125.60000000000001, 'dummy-lora-7b-rank-128-4'))]
ic| adapter_group: [['dummy-lora-7b-rank-16-3', 1.0],
                    ['dummy-lora-7b-rank-16-2', 1.0],
                    ['dummy-lora-7b-rank-16-1', 1.0],
                    ['dummy-lo

All adapters placed successfully for step 0 from time 0.016076426855815562 to 30.172266597782404
defaultdict(<class 'list'>, {'dummy-lora-7b-rank-16-3': [0], 'dummy-lora-7b-rank-16-2': [0], 'dummy-lora-7b-rank-16-1': [0], 'dummy-lora-7b-rank-16-0': [0], 'dummy-lora-7b-rank-16-4': [0], 'dummy-lora-7b-rank-8-1': [0], 'dummy-lora-7b-rank-8-4': [0], 'dummy-lora-7b-rank-8-2': [0], 'dummy-lora-7b-rank-8-0': [0], 'dummy-lora-7b-rank-64-0': [0, 3], 'dummy-lora-7b-rank-128-1': [1], 'dummy-lora-7b-rank-128-2': [1, 2], 'dummy-lora-7b-rank-128-3': [2], 'dummy-lora-7b-rank-128-0': [2, 3], 'dummy-lora-7b-rank-128-4': [3], 'dummy-lora-7b-rank-32-0': [3], 'dummy-lora-7b-rank-32-4': [3], 'dummy-lora-7b-rank-32-3': [3], 'dummy-lora-7b-rank-32-2': [3], 'dummy-lora-7b-rank-32-1': [3], 'dummy-lora-7b-rank-64-1': [3], 'dummy-lora-7b-rank-64-3': [3], 'dummy-lora-7b-rank-64-2': [3]})
defaultdict(<class 'list'>, {'dummy-lora-7b-rank-16-3': [1.0], 'dummy-lora-7b-rank-16-2': [1.0], 'dummy-lora-7b-rank-16-1': [1.

...

All adapters placed successfully for step 6 from time 180.00955675925917 to 210.11136117482278
Server renames detected: {1: 0, 2: 3}
Prev rank assigned instances (step 5): {8: [0], 32: [1], 64: [2], 128: [3]}
Curr rank assigned instances (step 6): {32: [0], 64: [1], 128: [2, 3]}
defaultdict(<class 'list'>, {'dummy-lora-7b-rank-32-2': [0], 'dummy-lora-7b-rank-32-4': [0], 'dummy-lora-7b-rank-32-3': [0], 'dummy-lora-7b-rank-32-1': [0], 'dummy-lora-7b-rank-32-0': [0], 'dummy-lora-7b-rank-8-4': [0], 'dummy-lora-7b-rank-8-1': [0], 'dummy-lora-7b-rank-16-3': [0, 0], 'dummy-lora-7b-rank-64-4': [0], 'dummy-lora-7b-rank-64-3': [0], 'dummy-lora-7b-rank-64-2': [0], 'dummy-lora-7b-rank-64-1': [0], 'dummy-lora-7b-rank-64-0': [0], 'dummy-lora-7b-rank-8-3': [0], 'dummy-lora-7b-rank-16-2': [0], 'dummy-lora-7b-rank-16-0': [0, 3], 'dummy-lora-7b-rank-128-1': [3], 'dummy-lora-7b-rank-128-3': [3], 'dummy-lora-7b-rank-128-2': [3, 3], 'dummy-lora-7b-rank-128-0': [3], 'dummy-lora-7b-rank-128-4': [3], 'dummy-l

 (16, 20.933333333333334, 'dummy-lora-7b-rank-16-4'))]
ic| adapters_of_rank: [(10, (32, 83.73333333333333, 'dummy-lora-7b-rank-32-2')),
                       (11, (32, 62.8, 'dummy-lora-7b-rank-32-4')),
                       (12, (32, 62.8, 'dummy-lora-7b-rank-32-1')),
                       (13, (32, 41.86666666666667, 'dummy-lora-7b-rank-32-0')),
                       (14, (32, 20.933333333333334, 'dummy-lora-7b-rank-32-3'))]
ic| adapters_of_rank: [(5, (64, 104.66666666666667, 'dummy-lora-7b-rank-64-3')),
                       (6, (64, 62.8, 'dummy-lora-7b-rank-64-4')),
                       (7, (64, 62.8, 'dummy-lora-7b-rank-64-1')),
                       (8, (64, 41.86666666666667, 'dummy-lora-7b-rank-64-0')),
                       (9, (64, 20.933333333333334, 'dummy-lora-7b-rank-64-2'))]
ic| adapters_of_rank: [(0, (128, 314.0, 'dummy-lora-7b-rank-128-4')),
                       (1, (128, 209.33333333333334, 'dummy-lora-7b-rank-128-0')),
                       (2, (128, 188

All adapters placed successfully for step 7 from time 210.11136117482278 to 240.06579478162047
Server renames detected: {0: 0, 1: 1, 2: 2, 3: 3}
Prev rank assigned instances (step 6): {32: [0], 64: [1], 128: [2, 3]}
Curr rank assigned instances (step 7): {16: [], 32: [0], 64: [1], 128: [2, 3]}
defaultdict(<class 'list'>, {'dummy-lora-7b-rank-32-2': [0], 'dummy-lora-7b-rank-32-4': [0], 'dummy-lora-7b-rank-32-1': [0], 'dummy-lora-7b-rank-32-0': [0], 'dummy-lora-7b-rank-32-3': [0], 'dummy-lora-7b-rank-128-2': [0, 3], 'dummy-lora-7b-rank-16-2': [0], 'dummy-lora-7b-rank-8-2': [0], 'dummy-lora-7b-rank-16-3': [0, 1], 'dummy-lora-7b-rank-64-3': [1], 'dummy-lora-7b-rank-64-4': [1], 'dummy-lora-7b-rank-64-1': [1], 'dummy-lora-7b-rank-64-0': [1], 'dummy-lora-7b-rank-64-2': [1], 'dummy-lora-7b-rank-8-0': [1], 'dummy-lora-7b-rank-16-1': [1], 'dummy-lora-7b-rank-16-0': [1], 'dummy-lora-7b-rank-8-3': [1], 'dummy-lora-7b-rank-8-1': [1], 'dummy-lora-7b-rank-16-4': [1], 'dummy-lora-7b-rank-128-4': [2], 

),
                       (6, (64, 104.66666666666667, 'dummy-lora-7b-rank-64-0')),
                       (7, (64, 41.86666666666667, 'dummy-lora-7b-rank-64-2')),
                       (8, (64, 20.933333333333334, 'dummy-lora-7b-rank-64-3')),
                       (9, (64, 0, 'dummy-lora-7b-rank-64-4'))]
ic| adapters_of_rank: [(0, (128, 209.33333333333334, 'dummy-lora-7b-rank-128-4')),
                       (1, (128, 188.4, 'dummy-lora-7b-rank-128-3')),
                       (2, (128, 146.53333333333333, 'dummy-lora-7b-rank-128-1')),
                       (3, (128, 146.53333333333333, 'dummy-lora-7b-rank-128-0')),
                       (4, (128, 104.66666666666667, 'dummy-lora-7b-rank-128-2'))]
ic| adapter_group: [['dummy-lora-7b-rank-32-3', 1.0],
                    ['dummy-lora-7b-rank-32-1', 1.0],
                    ['dummy-lora-7b-rank-32-0', 1.0],
                    ['dummy-lora-7b-rank-32-4', 1.0],
                    ['dummy-lora-7b-rank-32-2', 1.0]]
ic| adapter_group: 

All adapters placed successfully for step 8 from time 240.06579478162047 to 270.11525157262844
Server renames detected: {0: 0, 1: 1, 2: 2, 3: 3}
Prev rank assigned instances (step 7): {16: [], 32: [0], 64: [1], 128: [2, 3]}
Curr rank assigned instances (step 8): {32: [0], 64: [1], 128: [2, 3]}
defaultdict(<class 'list'>, {'dummy-lora-7b-rank-32-3': [0], 'dummy-lora-7b-rank-32-1': [0], 'dummy-lora-7b-rank-32-0': [0], 'dummy-lora-7b-rank-32-4': [0], 'dummy-lora-7b-rank-32-2': [0], 'dummy-lora-7b-rank-16-1': [0], 'dummy-lora-7b-rank-8-3': [0], 'dummy-lora-7b-rank-8-2': [0, 1], 'dummy-lora-7b-rank-64-1': [1], 'dummy-lora-7b-rank-64-0': [1], 'dummy-lora-7b-rank-64-2': [1], 'dummy-lora-7b-rank-64-3': [1], 'dummy-lora-7b-rank-8-1': [1], 'dummy-lora-7b-rank-8-0': [1], 'dummy-lora-7b-rank-16-4': [1], 'dummy-lora-7b-rank-16-2': [1, 3], 'dummy-lora-7b-rank-128-4': [2], 'dummy-lora-7b-rank-128-3': [2], 'dummy-lora-7b-rank-128-1': [2, 3], 'dummy-lora-7b-rank-128-0': [3], 'dummy-lora-7b-rank-128-2':